In [5]:
import pymupdf4llm, re
from langchain_chroma import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
import time, ollama, pickle
from pathlib import Path

In [4]:
image_dir = r'C:\Users\Prasanna Sakthivel\Desktop\Tasks\Spider\RAG\final\processing\images'
path = [
    'attention_is_all_you_need.pdf',
    'bert_Pre-training_of_deep_bidirectional_transformers_for_Language_understanding.pdf',
    'language_models_are_few-shot_learners.pdf',
    'llama2_open_foundation_and_fine-tuned_chat_models.pdf',
    'lora_low-rank_adaptation_of_large_language_models.pdf',
    'retrieval-augmented_generation_for_knowledge-intensive_nlp_tasks.pdf',
    'sentence-bert_sentence_embeddings_using_siamese_bert-networks.pdf'
]
md_list = []

def get_markdown(path, md_list):
    markdown = pymupdf4llm.to_markdown(
        doc=path,
        write_images=True,
        image_path=image_dir,
        image_format='png'
    )
    txt = markdown.split('\n')
    md_list.append(txt)
'''
for i in path:
    get_markdown(i, md_list)
print(md_list)
'''

'\nfor i in path:\n    get_markdown(i, md_list)\nprint(md_list)\n'

In [2]:
def get_text_before_chunking(txt):
    pattern = re.compile('\!\[(.*?)\]\((.*?\.(?:png|jpg|jpeg|gif))\)')
    image_dir = (r'C:\Users\Prasanna Sakthivel\Desktop\Tasks\Spider\RAG\final')
    count=0
    for i in range(len(txt)):
        if pattern.match(txt[i]):
            file_path = Path(image_dir+'\\'+pattern.match(txt[i]).group(2)).resolve()
            res = ollama.chat(
                model='moondream',  # <-- The lightest vision model available
                messages=[
                    {
                        'role': 'user',
                        'content': 'Describe the contents of this image clearly in markdown',
                        'images': [str(file_path)]
                    }
                ],
                options={
                    'temperature': 0.1
                }
            )
            txt[i] = "This is a figure description; Placeholder text for the figure:\n "+res.message.content
            count+=1
            if count%5==0:
                time.sleep(2)

    text_before_chunking = '\n'.join(txt)
    return text_before_chunking

In [4]:
bc_list = []
for i, txt in enumerate(md_list):
    k = get_text_before_chunking(txt)
    bc_list.append(k)
    print(f'done with pdf {i+1}')
    

done with pdf 1
done with pdf 2
done with pdf 3
done with pdf 4
done with pdf 5
done with pdf 6
done with pdf 7
done with pdf 8


In [1]:
import pickle, copy
with open('beforechunking.dat', 'rb') as file:
    k = pickle.load(file)
print(k.pop(3))
text_before_chunking = copy.deepcopy(k)


# **Language Models are Few-Shot Learners** 

**Tom B. Brown** _[∗]_ 

**Benjamin Mann** _[∗]_ **Nick Ryder** _[∗]_ **Melanie Subbiah** _[∗]_ 

**Jared Kaplan** _[†]_ **Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henighan** 

**Rewon Child Aditya Ramesh** 

**Daniel M. Ziegler Jeffrey Wu Clemens Winter** 

**Christopher Hesse Mark Chen** 

**Eric Sigler Mateusz Litwin Scott Gray** 

**Benjamin Chess** 

**Jack Clark Christopher Berner** 

**Sam McCandlish Alec Radford** 

**Ilya Sutskever Dario Amodei** 

## OpenAI 

## **Abstract** 

Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a ne

In [16]:
def get_chunks(pdf_name, text_before_chunking):
    headers_to_split_on = [
        ('#', 'Header 1'),
        ('##', 'Header 2'),
        ('###', 'Header 3')
    ]
    splitter_1 = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    splitter_2 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter_1.split_text(text_before_chunking)
    final_chunks = []

    for chunk in chunks:
        k = splitter_2.split_documents([chunk]) 
        pdf_name = pdf_name
        
        for doc in k:
            pro_st = 'source: {}\n'.format(pdf_name)
            for j in doc.metadata:
                pro_st += '{}: {}\n'.format(j, doc.metadata[j])
            doc.page_content = pro_st + doc.page_content
        final_chunks.extend(k)
    return final_chunks

fin_chunks = []
for i, p in enumerate(path):
    fin_chunks+=get_chunks(p, text_before_chunking[i])

for i in fin_chunks:
    print(i.page_content)
    print('-'*100)
    

source: attention_is_all_you_need.pdf
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
----------------------------------------------------------------------------------------------------
source: attention_is_all_you_need.pdf
Header 1: **Attention Is All You Need**
**Ashish Vaswani** _[∗]_ **Noam Shazeer** _[∗]_ **Niki Parmar** _[∗]_ **Jakob Uszkoreit** _[∗]_ Google Brain Google Brain Google Research Google Research `avaswani@google.com noam@google.com nikip@google.com usz@google.com`  
**Llion Jones** _[∗]_ **Aidan N. Gomez** _[∗†]_ **Łukasz Kaiser** _[∗]_ Google Research University of Toronto Google Brain `llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com`  
**Illia Polosukhin** _[∗‡]_  
```
illia.polosukhin@gmail.com
```
----------------------------------------------------------------------------------------------------
source: attention_is_all_you

In [20]:
def vectorize(fin_chunks):
    embed_model = OllamaEmbeddings(model='qwen3-embedding:4b')
    vector_store = Chroma.from_documents(
        documents=fin_chunks,
        embedding=embed_model,
        persist_directory='./chroma_db'
    )

vectorize(fin_chunks)

In [3]:

def init_session():
    global vector_store, chunks
    with open('chunks.dat', 'rb') as file:
        chunks = pickle.load( file)
    persist_dir = './chroma_db'
    vector_store = Chroma(
        embedding_function=OllamaEmbeddings(model='qwen3-embedding:4b', keep_alive=False),
        persist_directory=persist_dir
    )

def prompt(query: str) -> str:
    context_ls = vector_store.similarity_search(
        query=query,
        k=1
    )
    txt=''
    for i in context_ls:
        i.id = None
        index = chunks.index(i)
        txt += chunks[index-1].page_content+'\n'+chunks[index].page_content+'\n'+chunks[index+1].page_content+'\n'
    prm = f'''
    You are a helpful assistant. Don't hallucinate. 
    This is the question: {query}
 
    Answer the question with the given context:

    {txt}

    Please stick to the information given in the context, do not deviate or change
    '''
    llm = OllamaLLM(model='llama3.1', keep_alive=False)
    res = llm.stream(prm)
    for i in res:
        print(i, flush=True, end='')


In [ ]:
init_session()
prompt('what are encoders? explain in 500 words')

**Encoders: A Component of the Transformer Model**

In the context of the Transformer model architecture, an encoder is a crucial component that plays a significant role in processing input sequences. According to the paper "Attention Is All You Need" [1], the encoder is responsible for transforming input sequences into a sequence of vectors that can be used as input to the decoder.

**Definition and Functionality**

The encoder is composed of a stack of identical layers, denoted as _N_ = 6 in this case. Each layer consists of two sub-layers: a multi-head self-attention mechanism and a positionwise fully connected feed-forward network (FFN). The residual connection [11] is employed around each of the two sub-layers, followed by layer normalization [1].

**Encoder Architecture**

The encoder architecture can be summarized as follows:

*   Each layer has two main components:
    *   Multi-head self-attention mechanism: This component allows the model to attend to different parts of the i

In [41]:
prompt('explain the abstract of BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding in 500 words')

Here is a 500-word explanation of the abstract of BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding:

The abstract of the paper "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" introduces a new language representation model called BERT. BERT stands for Bidirectional Encoder Representations from Transformers, and it is designed to pre-train deep bidirectional representations from unlabeled text.

Unlike previous language representation models (Peters et al., 2018a; Radford et al., 2018), BERT is unique in its ability to jointly condition on both left and right context in all layers. This allows the pre-trained BERT model to be fine-tuned with just one additional output layer, enabling state-of-the-art results for a wide range of tasks without substantial task-specific architecture modifications.

The paper highlights that BERT is conceptually simple yet empirically powerful. It achieves new state-of-the-art results on elev

In [42]:
prompt('Explain the Helpfulness human evaluation results for Llama2-Chat')

According to Figure 1: Helpfulness human evaluation results for Llama2-Chat compared to other open-source and closed-source models. The helpfulness win rate of Llama2-Chat is shown in a bar graph with a positive relationship between the model's performance and its helpfulness win rate.

The exact numbers are not specified, but it mentions that the 95% confidence intervals for this evaluation are between 1% and 2%. 

Additionally, Figure 2 shows a comparison of Llama2-Chat with commercial-licensed baselines in terms of helpfulness and safety win rates. According to GPT4, when Llama2-Chat is better, the green area indicates that our model performs better according to GPT-4.

The human evaluation results show that Llama2-Chat has a positive relationship between its performance and helpfulness win rate compared to other models.

In [43]:
prompt('Explain what is LoRA in a 1000 words')

Here is an explanation of LoRA (Low-Rank Adaptation of Large Language Models) within the 1000-word limit, based on the provided sources:

**Introduction**

Large language models have revolutionized natural language processing tasks by achieving state-of-the-art results. However, fine-tuning these enormous models for specific downstream tasks is computationally expensive and often requires significant resources. To address this challenge, researchers proposed LoRA (Low-Rank Adaptation of Large Language Models), an efficient adaptation strategy that retains high model quality while reducing the computational cost.

**What is LoRA?**

LoRA is a low-rank adaptation method designed for large language models. It involves modifying the weight matrix (`W`) of the pre-trained model to adapt it to a specific downstream task. The core idea behind LoRA is to learn an additional small-rank matrix (`∆_W`) that can be easily added to the original weight matrix, thereby adapting the model without sign

In [9]:
init_session()
prompt("What is self attention and recurrence? Differentiate in 300 words")

Based on the provided context from "attention_is_all_you_need.pdf", here's the answer:

**Self-Attention and Recurrence: A Differentiation**

In this paper, self-attention is defined as an attention mechanism that relates different positions of a single sequence to compute a representation of the sequence. This means that self-attention focuses on internal dependencies within a single input or output position.

On the other hand, recurrence refers to the traditional sequential computation used in recurrent neural networks (RNNs). End-to-end memory networks use recurrent attention mechanisms instead of sequence-aligned recurrence and have been shown to perform well on tasks such as simple-language question answering and language modeling. This indicates that recurrence involves using a hidden state or memory that is updated at each time step, allowing the model to maintain a contextual representation over an input sequence.

The key differences between self-attention and recurrence are:

In [10]:
prompt("What problem does RAG solve?")

According to the provided context, RAG (Retrieval-Augmented Generation) solves the problem of "hallucination" in language generation. Hallucination refers to the model generating content that is not based on real factual knowledge, but rather on its own imagination. RAG's use of external knowledge sources, such as Wikipedia, helps to reduce hallucination and generate more factual and specific responses.

In [11]:
prompt('How does LoRA reduce training cost?')

According to the source "lora_low-rank_adaptation_of_large_language_models.pdf", LoRA (Low-Rank Adaptation) reduces training cost by greatly reducing the number of trainable parameters for downstream tasks. Specifically, it injects trainable rank decomposition matrices into each layer of the Transformer architecture, which can reduce the number of trainable parameters by 10,000 times compared to fine-tuning a large pre-trained model like GPT-3 175B with Adam. This reduction in parameters leads to lower GPU memory requirements (by 3 times) and likely decreases training time and costs as well.

In [16]:
prompt('differentiate GPT and BERT')

According to the provided source (bert_Pre-training_of_deep_bidirectional_transformers_for_Language_understanding.pdf), here are the differences between BERT and GPT:

1. **Training corpus**:
	* GPT is trained on the BooksCorpus (800M words).
	* BERT is trained on both the BooksCorpus (800M words) and Wikipedia (2,500M words).
2. **Pre-training vs Fine-tuning**:
	* Both models use a fine-tuning approach, but BERT learns [SEP], [CLS] and sentence A/B embeddings during pre-training.
	* GPT uses a sentence separator ([SEP]) and classifier token ([CLS]) which are only introduced at fine-tuning time.
3. **Training parameters**:
	* GPT was trained for 1M steps with a batch size of 32,000 words.
	* BERT was trained for 1M steps with a batch size of 128,000 words.
4. **Learning rate**:
	* GPT used the same learning rate of 5e-5 for all fine-tuning experiments.
	* BERT chooses a task-specific fine-tuning learning rate which performs the best on the development set.

These differences highlight 